# 🌊 AUV Swarm — Federated Learning & Reinforcement Learning (PPO)

Notebook tổng hợp toàn bộ dự án **AUV-Swarm-RFL**, chia thành **2 phần chính**:

| Phần | Nội dung |
|------|----------|
| **Phần 1 – Federated Learning** | 1A. FL tự code (FedAvg + Async Aggregation trên MNIST) |
| **Phần 2 – Reinforcement Learning** | 2A. Môi trường AUV Swarm (Gym)<br>2B. Mạng Actor-Critic & RolloutBuffer<br>2C. Thuật toán PPO (thiết kế modular, dễ thay thế bằng GRPO)<br>2D. Training loop PPO<br>2E. Evaluation & Plotting |

TODO: Trình bày bài toán lại : FL + RL: Pipeline - Lược đồ -> Mỗi input + output + Code


In [15]:
# Cài đặt thêm sb3 nếu cần thiết, giữ nguyên các thư viện khác
%pip install "protobuf>=3.20.3,<5" "tensorboard>=2.14,<2.17" stable-baselines3[extra]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, sys, copy, time, math, json
import numpy as np
import torch
# Torch CPU threading; =1 giúp ổn định tr   ên notebook/Windows, đổi lớn hơn nếu muốn tận dụng CPU.
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.datasets import MNIST
from collections import OrderedDict
import multiprocessing as mp
import matplotlib
matplotlib.use('Agg')  # Backend không cần cửa sổ GUI, phù hợp chạy notebook/remote.
import matplotlib.pyplot as plt


import gymnasium as gym
from gymnasium import spaces

# device: thiết bị chạy tensor/model chính (GPU nếu có, ngược lại CPU).
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# Global logging switches cho toàn pipeline.
# True  -> bật log module tương ứng.
# False -> giảm log để chạy nhanh và đỡ nhiễu output.
LOG_CFG = {
    'fl': False,   # Tắt log FL cho đỡ nhiễu
    'env': True,   # Environment step logs
    'ppo': False,  # Tắt log PPO (Actor / Critic / Loss)
    'train': True, # Training loop logs
    'eval': True,  # Evaluation logs
}

def log_if(enabled, tag, msg):
    # Hàm log có điều kiện để thống nhất format toàn notebook.
    if enabled:
        print(f"[{tag}] {msg}")

Device: cpu


---
# Phần 1: Federated Learning

## 1A. Federated Learning — Tự code (MNIST)

Bao gồm đầy đủ pipeline FL:
- **SimpleNN** — Mô hình CNN nhận dạng chữ số MNIST
- **DatasetSplit** & **get_mnist_data** — Chia dữ liệu IID cho N workers
- **local_train** — Hàm huấn luyện cục bộ tại mỗi AUV/worker
- **AsyncAggregator** — Bộ tổng hợp bất đồng bộ với cache (Lazy Nodes, Eq. 19)
- **FLSimulator** — Bộ điều phối FL chạy song song (multiprocessing)


In [17]:
# ─── 1A.1  Model CNN cho MNIST ───
class SimpleNN(nn.Module):
    """
    MLP 3 lớp cho MNIST (200-200-10) theo cấu trúc bài báo.
    (Giữ tên class SimpleNN để không phá vỡ các phần code dùng model này)
    """
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 200)
        self.fc2 = nn.Linear(200, 200)
        self.fc3 = nn.Linear(200, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def get_model_params(model):
    return OrderedDict({k: v.cpu().clone() for k, v in model.state_dict().items()})

def set_model_params(model, params):
    model.load_state_dict(params)

print(f"SimpleNN parameters: {sum(p.numel() for p in SimpleNN().parameters()):,}")


SimpleNN parameters: 199,210


In [18]:
# ─── 1A.2  Chia dữ liệu MNIST cho N workers (IID) ───
class DatasetSplit(Dataset):
    def __init__(self, dataset, idxs):
        self.dataset = dataset
        self.idxs = list(idxs)
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, item):
        return self.dataset[self.idxs[item]]

def get_mnist_data(num_users, iid=True):
    apply_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=apply_transform)
    test_dataset  = datasets.MNIST('./data', train=False, download=True, transform=apply_transform)

    dict_users = {}
    num_items = len(train_dataset) // num_users
    all_idxs = list(range(len(train_dataset)))
    for i in range(num_users):
        dict_users[i] = set(np.random.choice(all_idxs, num_items, replace=False))
        all_idxs = list(set(all_idxs) - dict_users[i])
    return train_dataset, test_dataset, dict_users

print("✓ Data utilities defined.")


✓ Data utilities defined.


In [19]:
# ─── 1A.3  Local Training (Worker) ───
def local_train(worker_id, global_params, dataset, idxs, num_epochs=1, lr=0.01, batch_size=128, verbose=False):
    """Chạy SGD cục bộ tại 1 worker. Trả về (updated_params, avg_loss)."""
    # worker_id: chỉ số worker trong FL.
    # global_params: trọng số global hiện tại nhận từ server/aggregator.
    # dataset + idxs: dữ liệu toàn cục và tập chỉ số mẫu thuộc worker.
    # num_epochs, lr, batch_size: siêu tham số local SGD.
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    model = SimpleNN().to(device)  # Đưa model lên GPU/CPU theo device.
    set_model_params(model, global_params)
    model.train()

    local_dataset = DatasetSplit(dataset, idxs)
    loader = DataLoader(local_dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.5)
    criterion = nn.CrossEntropyLoss()

    if verbose:
        log_if(True, 'LOCAL', f"worker={worker_id} | samples={len(local_dataset)} | epochs={num_epochs} | lr={lr}")

    epoch_loss = []
    for ep in range(num_epochs):
        batch_loss = []
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            # Mất mát phân loại cho MNIST.
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            batch_loss.append(loss.item())

        ep_loss = sum(batch_loss) / max(len(batch_loss), 1)
        epoch_loss.append(ep_loss)
        if verbose:
            log_if(True, 'LOCAL', f"worker={worker_id} | epoch={ep+1}/{num_epochs} | avg_loss={ep_loss:.6f}")

    final_loss = sum(epoch_loss) / max(len(epoch_loss), 1)
    if verbose:
        log_if(True, 'LOCAL', f"worker={worker_id} | done | final_loss={final_loss:.6f}")

    # Trả về params mới để server aggregate, kèm loss thống kê.
    return get_model_params(model), final_loss

print("✓ local_train defined.")

✓ local_train defined.


In [20]:
# ─── 1A.4  Async Aggregator (FedAvg + Cache cho Lazy Nodes) ───
class AsyncAggregator:
    def __init__(self, num_workers, initial_weights):
        self.num_workers = num_workers
        # Cache giữ trọng số mới nhất của TẤT CẢ workers
        self.worker_cache = {i: copy.deepcopy(initial_weights) for i in range(num_workers)}

    def update_and_aggregate(self, active_weights_dict):
        """
        Nhận {worker_id: weights} từ các Active Nodes.
        Cập nhật cache, rồi tính trung bình toàn bộ M nodes (Active + Lazy từ cache).
        """
        for w_id, weights in active_weights_dict.items():
            self.worker_cache[w_id] = copy.deepcopy(weights)

        averaged = {}
        for key in self.worker_cache[0].keys():
            averaged[key] = torch.zeros_like(self.worker_cache[0][key], dtype=torch.float32)
            for w_id in range(self.num_workers):
                averaged[key] += self.worker_cache[w_id][key]
            averaged[key] = torch.div(averaged[key], self.num_workers)
        return averaged

print("✓ AsyncAggregator defined.")


✓ AsyncAggregator defined.


In [21]:
# ─── 1A.5  Early Stopping ───
class EarlyStopping:
    def __init__(self, patience=20, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, accuracy):
        if self.best_score is None:
            self.best_score = accuracy
            return
        if accuracy > self.best_score + self.min_delta:
            self.best_score = accuracy
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def reset(self):
        self.best_score = None
        self.counter = 0
        self.early_stop = False

import asyncio
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from concurrent.futures.process import BrokenProcessPool

# ─── 1A.5  FL Simulator (Asynchronous Aggregation + Eq. 18/19/20) ───
def _worker_wrapper(args):
    # args = (worker_id, global_params, dataset, idxs, num_epochs, lr, batch_size)
    wid, gp, ds, idxs, ne, lr, bs = args
    return local_train(wid, gp, ds, idxs, ne, lr, bs)

async def simulate_worker(wid, gp, ds, idxs, ne, lr, bs, delay_time, executor):
    # Huấn luyện cục bộ worker trong executor (thread/process) rồi trả về trọng số local.
    loop = asyncio.get_running_loop()
    try:
        result = await loop.run_in_executor(executor, local_train, wid, gp, ds, idxs, ne, lr, bs)
    except (BrokenProcessPool, OSError):
        log_if(LOG_CFG.get('fl', False), 'FL', f"Process pool broke at worker={wid}, fallback to thread.")
        result = await asyncio.to_thread(local_train, wid, gp, ds, idxs, ne, lr, bs)
    # Có thể bật lại dòng sleep này nếu muốn mô phỏng trễ bất đồng bộ đúng theo T_LC.
    # await asyncio.sleep(delay_time)
    return wid, result


def evaluate_fl_model(model, test_dataset, batch_size=128):
    # Đánh giá nhanh accuracy global model trên test set.
    model.eval()
    loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

class FLSimulator:
    def __init__(self, num_workers=10, iid=True, executor_type='auto', max_executor_workers=None, verbose=None):
        self.num_workers = num_workers
        self.verbose = LOG_CFG.get('fl', True) if verbose is None else bool(verbose)
        print(f"Loading MNIST for {num_workers} workers...")
        self.train_dataset, self.test_dataset, self.dict_users = get_mnist_data(num_workers, iid)
        self.reset_episode_model()

        # Trên Windows/Jupyter, process pool cho hàm định nghĩa trong notebook thường kém ổn định.
        running_in_notebook = 'ipykernel' in sys.modules
        if executor_type == 'auto':
            if os.name == 'nt' or running_in_notebook:
                self.executor_type = 'thread'
            else:
                self.executor_type = 'thread' if torch.cuda.is_available() else 'process'
        else:
            self.executor_type = executor_type

        self.max_executor_workers = max_executor_workers or min(self.num_workers, (os.cpu_count() or 1))
        if self.executor_type == 'process':
            self.executor = ProcessPoolExecutor(max_workers=self.max_executor_workers)
        else:
            self.executor = ThreadPoolExecutor(max_workers=self.max_executor_workers)

        log_if(self.verbose, 'FL', f"executor_type={self.executor_type} | max_workers={self.max_executor_workers}")

    def close(self):
        executor_obj = getattr(self, 'executor', None)
        if executor_obj is not None:
            executor_obj.shutdown(wait=True, cancel_futures=True)
            self.executor = None
            log_if(self.verbose, 'FL', "executor closed")

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass

    def reset_episode_model(self):
        # Reset global model + cache aggregator cho mỗi episode RL.
        self.global_model = SimpleNN()
        global_params = get_model_params(self.global_model)
        self.aggregator = AsyncAggregator(self.num_workers, global_params)\n        self.early_stopping.reset()

    def run_rounds(self, num_rounds=5, lazy_factor=0.0, num_epochs=1, lr=0.01, batch_size=128, num_processes=4):
        accuracies, losses = [], []
        global_params = get_model_params(self.global_model)
        aggregator = AsyncAggregator(self.num_workers, global_params)
        w_prev, w_prev2 = copy.deepcopy(global_params), copy.deepcopy(global_params)

        for rnd in range(num_rounds):
            t0 = time.time()
            global_params = get_model_params(self.global_model)

            if rnd == 0:
                global_diff_sq = 0.0
            else:
                w_prev2 = copy.deepcopy(w_prev)
                w_prev = copy.deepcopy(global_params)
                # ||w_t - w_{t-1}||^2: biến thiên global model giữa 2 rounds.
                global_diff_sq = sum(torch.sum((w_prev[k] - w_prev2[k])**2).item() for k in w_prev)

            worker_args = [(i, copy.deepcopy(global_params), self.train_dataset,
                            list(self.dict_users[i]), num_epochs, lr, batch_size)
                           for i in range(self.num_workers)]
            results = [_worker_wrapper(a) for a in worker_args]

            # Ngưỡng lazy cũ theo lazy_factor (giữ để so sánh).
            threshold = float('inf') if lazy_factor == 0 else (1.0 / lazy_factor) * global_diff_sq
            active = {}
            for i, (w_local, loss) in enumerate(results):
                local_diff = sum(torch.sum((w_local[k] - global_params[k])**2).item() for k in w_local)
                if local_diff <= threshold and rnd > 0:
                    pass
                else:
                    active[i] = w_local

            if not active:
                rid = np.random.choice(self.num_workers)
                active[rid] = results[rid][0]

            new_global = aggregator.update_and_aggregate(active)
            set_model_params(self.global_model, new_global)
            acc = evaluate_fl_model(self.global_model, self.test_dataset)\n        self.early_stopping(acc)
            accuracies.append(acc)
            dt = time.time() - t0
            print(f"  Round {rnd+1:02d} | Acc: {acc*100:.2f}% | Active: {len(active)}/{self.num_workers} | {dt:.1f}s")

        return accuracies, losses

    def sync_run_step(self, delay_times, beta, force_active_mask, rnd, global_diff_sq,
                             num_epochs=1, lr=0.01, batch_size=64, N_total=None, gamma=0.01,
                             verbose=None):
        if verbose is None:
            verbose = self.verbose

        global_params = get_model_params(self.global_model)

        # Chạy đồng bộ hoàn toàn (tuần tự) để loại bỏ Deadlock ThreadPool/PyTorch
        all_results = {}
        all_local_norms = {}
        
        for wid in range(self.num_workers):
            # Chạy trực tiếp hàm local_train thay vì đưa vào ThreadPool
            w_local, loss = local_train(
                wid, copy.deepcopy(global_params), self.train_dataset,
                list(self.dict_users[wid]), num_epochs, lr, batch_size
            )
            
            # Tính toán local norm để xếp hạng
            local_count = float(len(self.dict_users[wid]))
            local_norm = sum(torch.sum((local_count * (w_local[k] - global_params[k]))**2).item() for k in w_local)
            
            all_results[wid] = (w_local, loss)
            all_local_norms[wid] = local_norm

        # β ∈ [0,1]: tỷ lệ lazy mong muốn; số active = ceil(M * (1-β)).
        active_count = max(1, int(np.ceil(self.num_workers * (1.0 - float(beta)))))

        # Chọn top-k worker có local_norm lớn nhất.
        sorted_workers = sorted(all_local_norms.keys(), key=lambda w: all_local_norms[w], reverse=True)
        active_set = set(sorted_workers[:active_count])

        # Ép active cho node lazy quá lâu.
        for wid in range(self.num_workers):
            force_active = bool(force_active_mask[wid]) if wid < len(force_active_mask) else False
            if force_active:
                active_set.add(wid)

        # Round đầu luôn cho full participation.
        if rnd == 0:
            active_set = set(range(self.num_workers))

        active_indices = list(active_set)

        # Aggregate chỉ từ active workers, lazy workers giữ cache cũ trong AsyncAggregator.
        active_weights = {wid: all_results[wid][0] for wid in active_indices if wid in all_results}
        if active_weights:
            global_params = self.aggregator.update_and_aggregate(active_weights)
            set_model_params(self.global_model, global_params)

        acc = evaluate_fl_model(self.global_model, self.test_dataset)\n        self.early_stopping(acc)

        if verbose:
            log_if(True, 'FL', f"done rnd={rnd+1} | active={len(active_indices)}/{self.num_workers} | acc={acc*100:.2f}%")

        return acc, active_indices, self.early_stopping.early_stop\n

---
# Phần 2: Reinforcement Learning

## 2A. Môi trường AUV Swarm (OpenAI Gym)

Mô phỏng bầy đàn AUV với bài toán tối ưu **năng lượng + độ trễ** trong FL dưới nước:
- **State**: tham số điều khiển ở bước trước $(p_m, f_m, p_L, f_L, b)$
- **Action**: tham số điều khiển ở bước hiện tại (liên tục, [-1, 1])
- **Reward**: $-\text{Cost}$ (Latency + Energy + Penalty)


In [32]:
class AUVSwarmEnv(gym.Env):
    def __init__(self, M=9, debug=None, fl_sim=None, w_acc=100.0, w_energy=1.0, w_time=1.0):
        super(AUVSwarmEnv, self).__init__()
        self.M = M  # Số lượng follower AUV trong bầy.
        self.debug = LOG_CFG.get('env', True) if debug is None else bool(debug)  # Bật/tắt log chi tiết của môi trường.

        if fl_sim is None:
            raise ValueError("fl_sim is required for co-design mode. Please pass an initialized FLSimulator.")
        self.fl_sim = fl_sim  # Bộ mô phỏng FL được environment gọi ở mỗi step.
        if getattr(self.fl_sim, 'num_workers', self.M) != self.M:
            raise ValueError(f"fl_sim.num_workers ({self.fl_sim.num_workers}) must match M ({self.M}).")

        # Trọng số của hàm mục tiêu co-design.
        self.w_acc = float(w_acc)      # Trọng số cho accuracy.
        self.w_energy = float(w_energy)  # Trọng số cho energy.
        self.w_time = float(w_time)    # Trọng số cho time.

        # Tham số vật lý / truyền thông / tính toán.
        self.k_cap = 1.25e-26  # Hệ số năng lượng tính toán CMOS.
        self.d_factor = 3.0     # Bậc phụ thuộc của năng lượng theo tần số CPU.
        self.B_Um = 10e3;  self.B_D = 10e3  # Băng thông uplink và downlink (Hz).
        self.N_m = 4224;   self.c_m = 10000  # Số mẫu local mỗi worker và số chu kỳ CPU/mẫu.
        self.c_0 = 50;     self.c_0_L = 50   # Chu kỳ CPU của bước aggregate và leader update.
        self.w_m_size = 1594 * 64  # Kích thước model truyền từ follower (bit).
        self.w_size = 1594 * 64    # Kích thước model truyền/cập nhật của leader (bit).
        self.y = 64.0              # Overhead gói truyền (bit).
        self.p_max = 0.2           # Công suất phát cực đại (W).
        self.gamma_env = 0.01      # Hệ số phụ trợ của môi trường.
        self.f_max = 0.4e9         # Tần số CPU cực đại (Hz).
        self.f_min = 0.2e9         # Tần số CPU cực tiểu (Hz).
        self.E_thd_m = 0.12       # Ngưỡng năng lượng trung bình cho follower.
        self.E_thd_L = 1.0        # Ngưỡng năng lượng trung bình cho leader.
        self.freq_acoustic = 30    # Tần số âm học trong kênh nước.
        self.s = 0.5               # Tham số môi trường cho mô hình noise.
        self.D_m = np.ones(self.M) * 20.0  # Khoảng cách truyền âm học của từng follower (m).

        # Ngân sách pin.
        self.max_battery_m = 10.0  # Pin tối đa của follower.
        self.max_battery_L = 20.0  # Pin tối đa của leader.

        # Action: p_m(M), f_m(M), p_L(1), f_L(1), beta(1) => 2M+3.
        self.action_dim = 2 * self.M + 3  # Số chiều action.
        self.state_dim = 3 * self.M + 6    # Số chiều observation/state.
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.action_dim,), dtype=np.float32)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(self.state_dim,), dtype=np.float32)

        self.current_step = 0   # Bộ đếm bước hiện tại trong episode.
        self.max_steps = 50     # 1 step = 1 round FL, tối đa 50 round/episode.

        self.current_acc = 0.0  # Accuracy hiện tại của global model.
        self.prev_acc = 0.0     # Accuracy của step trước để tính delta_acc.
        self.global_diff_sq = 0.0   # ||w_t - w_{t-1}||^2 của global model.
        self.global_diff_norm = 0.0 # Phiên bản chuẩn hóa của global_diff_sq.
        self.prev_global_params = None  # Snapshot tham số global ở step trước.

        self.lazy_consecutive = np.zeros(self.M, dtype=np.int32)  # Đếm số step lazy liên tiếp của từng node.
        self._phase_cache = None  # Cache tạm giữa apply_action và finalize_step.

        self.N_f = self._noise(self.freq_acoustic)  # Mật độ nhiễu âm học tại tần số đã chọn.
        self.A_D = self._attenuation(self.D_m[0], self.freq_acoustic)  # Hệ số suy hao theo khoảng cách.
        self.channel_gain = 1.0 / (self.A_D * self.N_f)  # Gain xấp xỉ của kênh nước.

    def _attenuation(self, D, f):
        # Mô hình suy hao âm học theo khoảng cách D và tần số f.
        log_a = 0.11*f**2/(1+f**2) + 44*f**2/(4100+f**2) + 2.75e-4*f**2 + 0.003
        a_f = 10**(log_a/10.0)  # Suy hao theo tần số.
        return (D**1.5) * (a_f**(D/1000.0))

    def _noise(self, f):
        # Mô hình nhiễu tổng hợp (shipping / wind / thermal / jitter).
        log_NJ = 17 - 30*math.log10(f) if f > 0 else 0  # Nhiễu tàu thuyền.
        log_Ns = 40 + 20*(self.s-0.5) + 26*math.log10(f) - 60*math.log10(f+0.03)  # Nhiễu gió/bề mặt.
        log_Nw = 50 + 20*math.log10(f) - 40*math.log10(f+0.4)  # Nhiễu sóng / water agitation.
        log_Nth = -15 + 20*math.log10(f)  # Nhiễu nhiệt.
        N = sum(10**(x/10.0) for x in [log_NJ, log_Ns, log_Nw, log_Nth])
        return N * 1e-12  # Đưa về thang phù hợp với mô hình kênh.

    def _sync_run_fl_round(self, delay_times, beta, force_active_mask):
        # Chạy 1 round FL theo cách đồng bộ để tránh vấn đề thread / CUDA / notebook.
        return self.fl_sim.sync_run_step(
            delay_times=delay_times,
            beta=beta,
            force_active_mask=force_active_mask,
            rnd=self.current_step,
            global_diff_sq=self.global_diff_sq,
        )

    def _compute_global_diff_sq(self, new_params):
        if self.prev_global_params is None:
            return 0.0
        return float(sum(torch.sum((new_params[k] - self.prev_global_params[k])**2).item() for k in new_params))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.lazy_consecutive = np.zeros(self.M, dtype=np.int32)
        self._phase_cache = None

        self.battery_m = np.ones(self.M) * self.max_battery_m  # Pin của từng follower.
        self.battery_L = self.max_battery_L  # Pin của leader.

        self.current_acc = 0.0
        self.prev_acc = 0.0
        self.global_diff_sq = 0.0
        self.global_diff_norm = 0.0

        # Mỗi episode RL bắt đầu lại global model từ đầu.
        self.fl_sim.reset_episode_model()
        self.prev_global_params = copy.deepcopy(get_model_params(self.fl_sim.global_model))

        self.state = np.concatenate([
            np.full(self.M, 0.5, dtype=np.float32),  # p_m chuẩn hóa ban đầu.
            np.full(self.M, 0.5, dtype=np.float32),  # f_m chuẩn hóa ban đầu.
            np.array([0.5, 0.5, 0.5], dtype=np.float32),  # p_L, f_L, beta ban đầu.
            np.full(self.M, 1.0, dtype=np.float32),  # Pin follower ban đầu.
            np.array([1.0, self.current_acc, self.global_diff_norm], dtype=np.float32),  # Pin leader, acc, drift.
        ]).astype(np.float32)
        return self.state, {}

    def apply_action(self, action):
        # Action chuẩn hóa [-1, 1] được map sang miền vật lý.
        action = np.clip(action, -1.0, 1.0)
        p_m = np.clip((action[:self.M] + 1) / 2 * self.p_max, 0.01, self.p_max)  # Công suất phát của follower m.
        f_m = self.f_min + (action[self.M:2*self.M] + 1) / 2 * (self.f_max - self.f_min)  # Tần số CPU của follower m.
        p_L = np.clip((action[2*self.M] + 1) / 2 * self.p_max, 0.01, self.p_max)  # Công suất phát của leader.
        f_L = self.f_min + (action[2*self.M+1] + 1) / 2 * (self.f_max - self.f_min)  # Tần số CPU của leader.
        beta = float((action[2*self.M+2] + 1) / 2)  # Tỷ lệ lazy mong muốn trong khoảng [0, 1].

        alive_mask = self.battery_m > 0.01  # Mặt nạ node còn pin.

        # T_LC[m] = N_m*c_m / f_m[m]: thời gian local compute của follower m.
        T_LC = (self.N_m * self.c_m) / f_m
        # E_Cp[m] = k * f_m[m]^(d-1) * (N_m*c_m): năng lượng tính toán của follower m.
        E_Cp = self.k_cap * (f_m ** (self.d_factor - 1.0)) * (self.N_m * self.c_m)

        force_active_mask = (self.lazy_consecutive >= 5)  # Ép active nếu node lazy quá lâu.
        delay_times = T_LC.astype(np.float32)  # Độ trễ local compute đưa vào FL simulator.

        self._phase_cache = {
            'alive_mask': alive_mask,
            'T_LC': T_LC,
            'E_Cp': E_Cp,
            'f_m': f_m,
            'p_m': p_m,
            'p_L': p_L,
            'f_L': f_L,
            'beta': beta,
        }

        phase_params = {
            'p_m': p_m,
            'f_m': f_m,
            'p_L': p_L,
            'f_L': f_L,
            'alive_mask': alive_mask,
        }
        return delay_times, beta, force_active_mask, phase_params

    def finalize_step(self, active_indices, p_m, f_L, p_L, is_converged):
        T_LC = self._phase_cache['T_LC']
        E_Cp = self._phase_cache['E_Cp']
        f_m = self._phase_cache['f_m']
        beta = self._phase_cache['beta']
        alive_mask = self._phase_cache['alive_mask']

        active_mask = np.zeros(self.M, dtype=bool)  # Mặt nạ các node thực sự active.
        for idx in active_indices:
            if 0 <= idx < self.M and alive_mask[idx]:
                active_mask[idx] = True

        # FIX LỖI 1: Node inactive/lazy không tiêu thụ năng lượng tính toán trong round này.
        E_Cp[~active_mask] = 0.0

        penalty = 0.0  # Tổng penalty của step.
        timeout_extra = 0.0  # Phụ phí thời gian nếu bị ép active.

        # Phạt node hết pin.
        dead_count = (~alive_mask).sum()  # Số node đã chết pin.
        pen_dead = 0.0
        if dead_count > 0:
            pen_dead = float(dead_count * 1.0)  # Mỗi node chết pin bị phạt 1.0.
            penalty += pen_dead
            log_if(self.debug, 'ENV-PENALTY', f"Dead battery: {dead_count} nodes. Penalty += {pen_dead}")

        # Nếu không có node nào active thì ép 1 node active và phạt.
        pen_no_active = 0.0
        if active_mask.sum() == 0:
            pen_no_active = 10.0
            penalty += pen_no_active
            timeout_extra += 10.0
            log_if(self.debug, 'ENV-PENALTY', "No active nodes. Forced 1. Penalty += 10.0")
            alive_indices = np.where(alive_mask)[0]
            if len(alive_indices) > 0:
                forced_idx = int(np.random.choice(alive_indices))
            else:
                forced_idx = int(np.random.choice(self.M))
            active_mask[forced_idx] = True

        # R_U[m] = B_Um * log2(1 + p_m[m] * g / B_Um): tốc độ dữ liệu uplink của follower m.
        R_U = self.B_Um * np.log2(1 + p_m * self.channel_gain / self.B_Um)
        R_U = np.clip(R_U, 1e-6, np.inf)  # Chặn dưới để tránh chia cho 0.

        # T_LU[m] = (w_m_size + y) / R_U[m]: thời gian upload mô hình của follower m.
        T_LU = np.zeros(self.M, dtype=np.float64)
        T_LU[active_mask] = (self.w_m_size + self.y) / R_U[active_mask]

        # E_Cm[m] = p_m[m] * T_LU[m]: năng lượng truyền uplink của follower m.
        E_Cm = np.zeros(self.M, dtype=np.float64)
        E_Cm[active_mask] = p_m[active_mask] * T_LU[active_mask]

        # T_GA: thời gian aggregate của leader, tỷ lệ với số node active.
        T_GA = self.c_0 * (self.w_m_size * active_mask.sum()) / f_L

        # FIX LỖI 2: Thời gian leader update (T_GU) không được bị triệt tiêu.
        # T_GU = (c_0_L * w_size) / f_L: thời gian leader cập nhật tham số global.
        T_GU = (self.c_0_L * self.w_size) / f_L

        # R_D[m] = B_D * log2(1 + p_L * g / B_D): tốc độ dữ liệu downlink của leader.
        R_D = self.B_D * np.log2(1 + p_L * self.channel_gain / self.B_D)

        # T_GD = (w_size + y) / R_D: thời gian leader gửi model xuống các follower.
        T_GD = np.max((self.w_size + self.y) / R_D)

        # FIX LỖI 3: Chỉ lấy max thời gian compute/upload của các node active.
        max_compute_time = float(np.max(T_LC[active_mask])) if active_mask.sum() > 0 else 0.0
        max_upload_time = float(np.max(T_LU[active_mask])) if active_mask.sum() > 0 else 0.0

        T_total = float(max_compute_time + max_upload_time + np.max(T_GA + T_GU + T_GD) + timeout_extra)  # Tổng thời gian round.

        # E_CpL = k * f_L^(d-1) * (c_0 * w_m_size * |A| + c_0_L * w_size): năng lượng tính toán của leader.
        E_CpL = self.k_cap * (f_L ** (self.d_factor - 1.0)) * (self.c_0 * self.w_m_size * active_mask.sum() + self.c_0_L * self.w_size)

        # E_CL = p_L * T_GD: năng lượng truyền của leader.
        E_CL = p_L * T_GD

        # Tổng năng lượng của follower và leader.
        E_m_total = E_Cp + E_Cm  # Năng lượng từng follower.
        E_L_total = E_CpL + E_CL  # Năng lượng của leader.

        # Tính công suất trung bình trên toàn round.
        P_m_avg = E_m_total / max(T_total, 1e-8)  # Công suất trung bình của từng follower.
        P_L_avg = E_L_total / max(T_total, 1e-8)  # Công suất trung bình của leader.

        # CẢI TIẾN 4: Tăng sức nặng hình phạt để tránh vi phạm năng lượng bị bỏ qua.
        m_violation_count = (P_m_avg > self.E_thd_m).sum()  # Số follower vượt ngưỡng.
        pen_e_m = 0.0
        if m_violation_count > 0:
            pen_e_m = float(m_violation_count * 10.0)  # Phạt 10.0 cho mỗi node follower vi phạm.
            penalty += pen_e_m
            log_if(self.debug, 'ENV-PENALTY', f"Follower energy threshold violated by {m_violation_count} nodes. Penalty += {pen_e_m}")

        pen_e_l = 0.0
        if P_L_avg > self.E_thd_L:
            pen_e_l = 10.0  # Phạt cố định cho leader nếu vượt ngưỡng.
            penalty += pen_e_l
            log_if(self.debug, 'ENV-PENALTY', f"Leader energy threshold violated. Penalty += 10.0")

        # E_total: tổng năng lượng tiêu thụ của cả bầy trong round.
        phi_weight = 1.0
        chi_weight = 1.0
        E_total = float(phi_weight * (np.sum(E_Cp) + E_CpL) + chi_weight * (np.sum(E_Cm) + E_CL))

        # Trừ pin của follower và leader sau khi đã tiêu thụ năng lượng.
        self.battery_m -= E_m_total
        self.battery_m = np.clip(self.battery_m, 0.0, self.max_battery_m)  # Giới hạn pin follower trong [0, max].
        self.battery_L -= E_L_total
        self.battery_L = max(0.0, self.battery_L)  # Không cho pin leader âm.

        self.lazy_consecutive += 1  # Mặc định tăng lazy counter cho tất cả node.
        self.lazy_consecutive[active_mask] = 0  # Reset lazy counter với node active.

        # cost_normalized: chi phí đã chuẩn hóa để dùng trong reward.
        cost_time_norm = T_total / 10.0
        cost_energy_norm = E_total / 0.5
        cost_penalty = penalty
        cost_normalized = cost_time_norm + cost_energy_norm + cost_penalty

        # ---------------------------------------------------------
        # REWARD RESHAPING: Soft Clipping Gain bằng np.tanh
        # ---------------------------------------------------------
        delta_acc = float(self.current_acc) - self.prev_acc  # Mức tăng accuracy ở step hiện tại.
        self.prev_acc = float(self.current_acc)              # Lưu accuracy hiện tại cho step kế tiếp.

        w_energy = 2.0   # Trọng số năng lượng trong resource_cost.
        w_time = 10.0    # Trọng số thời gian trong resource_cost.
        resource_cost = (w_energy * float(cost_energy_norm)) + (w_time * float(cost_time_norm))  # Chi phí tài nguyên tổng hợp.
        resource_cost = max(resource_cost, 1e-4)  # Tránh chia cho 0.

        # Chống spike đầu episode: nén delta_acc + warmup theo step.
        warmup_steps = 8.0
        warmup_scale = min(1.0, float(self.current_step + 1) / warmup_steps)  # Hệ số warmup tăng dần từ 0 -> 1.
        delta_ref = 0.05  # Mốc chuẩn cho mức tăng accuracy mạnh.
        delta_smoothed = float(delta_acc) * warmup_scale  # Delta accuracy đã làm mượt.
        scaled_delta = 100.0 * np.tanh(delta_smoothed / delta_ref)  # Soft-clip delta về thang [-100, 100].
        noise_tolerance = -1.0  # Ngưỡng chấp nhận nhiễu nhẹ.
        REWARD_MULTIPLIER = 20.0  # Hệ số nhân reward chính.

        if scaled_delta >= noise_tolerance:
            effective_delta = max(0.0, scaled_delta)  # Chỉ giữ phần tăng dương có ích.
            base_gain = effective_delta / resource_cost  # Gain cơ bản theo hiệu quả tăng accuracy / chi phí.
            efficiency_bonus = 1.0 / resource_cost  # Thưởng thêm cho phương án tiết kiệm tài nguyên.
            raw_ar_gain = (base_gain + (0.5 * efficiency_bonus)) * REWARD_MULTIPLIER  # Gain thô trước khi clip.

            # REWARD RESHAPING: Soft Clipping bằng tanh để tránh reward bùng nổ.
            ar_gain = 100.0 * np.tanh(raw_ar_gain / 50.0)  # Giới hạn reward dần về 100.
        else:
            raw_ar_gain = scaled_delta * resource_cost  # Trường hợp delta xấu thì reward âm theo chi phí.
            ar_gain = raw_ar_gain * (REWARD_MULTIPLIER / 2.0)  # Phạt mạnh hơn khi delta kém.

        # Ràng buộc học tập cơ bản.
        learning_penalty = 0.0
        if self.current_step >= 5 and self.current_acc < 0.20:
            learning_penalty += 50.0  # Phạt nếu học quá chậm ở giai đoạn giữa episode.

        if delta_acc < -0.02:
            learning_penalty += 10.0  # Phạt nếu accuracy tụt mạnh.

        # Final Reward.
        reward = ar_gain - float(cost_penalty) - learning_penalty

        cost_raw = T_total + E_total + penalty  # Chi phí vật lý gốc chưa chuẩn hóa.

        if self.debug:
            print(f"\n--- THONG KE STEP (Env Co-design) ---")
            print(f"Acc hien tai: {self.current_acc:.4f} | Delta acc: {delta_acc:.6f} | Reward = {reward:.4f}")
            print(f"Warmup scale: {warmup_scale:.3f} | Scaled delta: {scaled_delta:.3f}")
            print(f"AR Gain (Reshaped): {ar_gain:.2f} | Raw Gain: {raw_ar_gain if 'raw_ar_gain' in locals() else 0:.2f}")
            print(f"Penalty: {cost_penalty:.2f} | Learning Penalty: {learning_penalty:.2f}")
            print(f"Tong thoi gian vong: T_total = {T_total:.4f} s | E_total = {E_total:.4f}")
            print("-------------------------------------\n")

        pin_leader = float(self.battery_L / self.max_battery_L)  # Pin leader đã chuẩn hóa về [0, 1].
        self.state = np.concatenate([
            p_m / self.p_max,  # Công suất follower đã chuẩn hóa.
            (f_m - self.f_min) / (self.f_max - self.f_min),  # Tần số follower đã chuẩn hóa.
            [p_L / self.p_max],  # Công suất leader đã chuẩn hóa.
            [(f_L - self.f_min) / (self.f_max - self.f_min)],  # Tần số leader đã chuẩn hóa.
            [beta],  # Tỷ lệ lazy đã chuẩn hóa.
            self.battery_m / self.max_battery_m,  # Pin follower đã chuẩn hóa.
            [pin_leader],  # Pin leader đã chuẩn hóa.
            [float(self.current_acc)],  # Accuracy hiện tại.
            [float(self.global_diff_norm)],  # Độ lệch model global đã chuẩn hóa.
        ]).astype(np.float32)

        self.current_step += 1  # Sang bước tiếp theo.
        early_stop = dead_count >= (self.M / 2)  # Dừng sớm nếu quá nửa node đã chết pin.
        done = bool(self.current_step >= self.max_steps or early_stop or is_converged)

        info = {
            'cost': cost_raw,
            'cost_normalized': cost_normalized,
            'cost_time_norm': float(cost_time_norm),
            'cost_energy_norm': float(cost_energy_norm),
            'cost_penalty': float(cost_penalty),
            'reward': float(reward),
            'time': T_total,
            'comm_time': float(np.max(T_LU[active_mask]) + T_GD) if active_mask.sum() > 0 else float(T_GD),
            'energy': E_total,
            'penalty': penalty,
            'participating': int(active_mask.sum()),
            'beta': float(beta),
            'alive': int(alive_mask.sum()),
            'battery_min': float(self.battery_m.min()),
            'current_accuracy': float(self.current_acc),
            'delta_accuracy': float(delta_acc),
            'delta_scaled': float(scaled_delta),
            'delta_warmup_scale': float(warmup_scale),
            'global_diff_sq': float(self.global_diff_sq),
            'global_diff_norm': float(self.global_diff_norm),
        }
        self._phase_cache = None  # Xóa cache tạm sau khi finalize xong.
        return self.state, reward, done, False, info

    def step(self, action):
        # 1) Giải action -> tham số vật lý.
        delay_times, beta, force_mask, params = self.apply_action(action)

        # 2) Chạy 1 round FL theo cách đồng bộ trong Gym step.
        acc, active_indices, is_converged = self._sync_run_fl_round(delay_times, beta, force_mask)

        # 3) Cập nhật trạng thái học FL: accuracy và độ lệch global model.
        self.current_acc = float(acc)
        new_global_params = copy.deepcopy(get_model_params(self.fl_sim.global_model))
        self.global_diff_sq = self._compute_global_diff_sq(new_global_params)
        self.global_diff_norm = float(np.tanh(self.global_diff_sq))
        self.prev_global_params = new_global_params

        # 4) Đưa active set thực tế vào bước finalize để tính cost và reward.
        return self.finalize_step(active_indices, params['p_m'], params['f_L'], params['p_L'], is_converged)\n

In [23]:
class AUVSwarmEnvVerbose(AUVSwarmEnv):
    """Ban giu tuong thich cho che do debug; toan bo logic co-design dung lai tu class cha."""
    pass

print("✓ AUVSwarmEnvVerbose defined (inherits co-design env).")

✓ AUVSwarmEnvVerbose defined (inherits co-design env).


---
## 🧪 Kiểm tra môi trường (Environment Testing)

Trước khi chạy training, ta **bắt buộc** phải kiểm tra xem `AUVSwarmEnv` có hoạt động đúng chuẩn Gymnasium không.

3 cách test:
1. **check_env** — Kiểm tra cấu trúc (Action/Observation Space, dtype, bounds)
2. **Random Agent** — Chạy thử 2 episode với hành động ngẫu nhiên (Sanity Check)
3. **Edge Cases** — Test hành động cực đoan (Max/Min power)


In [ ]:
# # ─── Cách 4: In toàn bộ từng thành phần trong finalize_step ───
# # Cell này đi theo đúng pipeline step của env nhưng tách manual để soi đầy đủ biến trung gian.

# if 'fl_sim_test' not in globals() or fl_sim_test is None:
#     print("Khởi tạo mới fl_sim_test...")
#     fl_sim_test = FLSimulator(num_workers=9, executor_type='thread', verbose=False)


# def _calc_finalize_components(env_obj, active_indices, p_m, f_L, p_L):
#     """Tái hiện các phép tính trong finalize_step để in từng thành phần."""
#     T_LC = env_obj._phase_cache['T_LC']
#     E_Cp = env_obj._phase_cache['E_Cp']
#     f_m = env_obj._phase_cache['f_m']
#     beta = env_obj._phase_cache['beta']
#     alive_mask = env_obj._phase_cache['alive_mask']

#     active_mask = np.zeros(env_obj.M, dtype=bool)
#     for idx in active_indices:
#         if 0 <= idx < env_obj.M and alive_mask[idx]:
#             active_mask[idx] = True

#     dead_count = int((~alive_mask).sum())
#     pen_dead = float(dead_count * 1.0) if dead_count > 0 else 0.0

#     pen_no_active = 0.0
#     timeout_extra = 0.0
#     if active_mask.sum() == 0:
#         pen_no_active = 10.0
#         timeout_extra = 10.0

#     R_U = env_obj.B_Um * np.log2(1 + p_m * env_obj.channel_gain / env_obj.B_Um)
#     R_U = np.clip(R_U, 1e-6, np.inf)

#     T_LU = np.zeros(env_obj.M, dtype=np.float64)
#     T_LU[active_mask] = (env_obj.w_m_size + env_obj.y) / R_U[active_mask]

#     E_Cm = np.zeros(env_obj.M, dtype=np.float64)
#     E_Cm[active_mask] = p_m[active_mask] * T_LU[active_mask]

#     T_GA = env_obj.c_0 * (env_obj.w_m_size * active_mask.sum()) / f_L
#     T_GU = (env_obj.c_0_L * env_obj.w_size) / f_L

#     R_D = env_obj.B_D * np.log2(1 + p_L * env_obj.channel_gain / env_obj.B_D)
#     T_GD = np.max((env_obj.w_size + env_obj.y) / R_D)

#     max_compute_time = float(np.max(T_LC[alive_mask])) if alive_mask.sum() > 0 else 0.0
#     max_upload_time = float(np.max(T_LU[active_mask])) if active_mask.sum() > 0 else 0.0
#     T_total = float(max_compute_time + max_upload_time + np.max(T_GA + T_GU + T_GD) + timeout_extra)

#     E_CpL = env_obj.k_cap * (f_L ** (env_obj.d_factor - 1.0)) * (
#         env_obj.c_0 * env_obj.w_m_size * active_mask.sum() + env_obj.c_0_L * env_obj.w_size
#     )
#     E_CL = p_L * T_GD

#     E_m_total = E_Cp + E_Cm
#     E_L_total = E_CpL + E_CL

#     P_m_avg = E_m_total / max(T_total, 1e-8)
#     P_L_avg = E_L_total / max(T_total, 1e-8)

#     m_violation_count = int((P_m_avg > env_obj.E_thd_m).sum())
#     pen_e_m = float(m_violation_count * 10.0) if m_violation_count > 0 else 0.0
#     pen_e_l = 10.0 if P_L_avg > env_obj.E_thd_L else 0.0

#     penalty = pen_dead + pen_no_active + pen_e_m + pen_e_l

#     phi_weight = 1.0
#     chi_weight = 1.0
#     E_total = float(phi_weight * (np.sum(E_Cp) + E_CpL) + chi_weight * (np.sum(E_Cm) + E_CL))

#     cost_time_norm = T_total / 10.0
#     cost_energy_norm = E_total / 0.5
#     cost_penalty = penalty
#     cost_normalized = cost_time_norm + cost_energy_norm + cost_penalty

#     delta_acc = float(env_obj.current_acc) - env_obj.prev_acc
#     warmup_steps = 8.0
#     warmup_scale = min(1.0, float(env_obj.current_step + 1) / warmup_steps)
#     delta_ref = 0.03
#     delta_smoothed = float(delta_acc) * warmup_scale
#     scaled_delta = 100.0 * np.tanh(delta_smoothed / delta_ref)
#     noise_tolerance = -1.0

#     w_energy = 2.0
#     w_time = 10.0
#     resource_cost = (w_energy * float(cost_energy_norm)) + (w_time * float(cost_time_norm))
#     resource_cost = max(resource_cost, 1e-4)

#     REWARD_MULTIPLIER = 20.0
#         raw_ar_gain = (base_gain + (0.5 * efficiency_bonus)) * REWARD_MULTIPLIER
#         ar_gain = 100.0 * np.tanh(raw_ar_gain / 50.0)
#         base_gain = effective_delta / resource_cost
#         efficiency_bonus = 1.0 / resource_cost
#         raw_ar_gain = base_gain + (0.5 * efficiency_bonus)
#         ar_gain = raw_ar_gain * REWARD_MULTIPLIER
#     else:
#         raw_ar_gain = scaled_delta * resource_cost
#         ar_gain = raw_ar_gain * (REWARD_MULTIPLIER / 2.0)

#     learning_penalty = 0.0
#     if env_obj.current_step >= 5 and env_obj.current_acc < 0.20:
#         learning_penalty += 50.0
#     if delta_acc < -0.02:
#         learning_penalty += 10.0

#     reward = ar_gain - float(cost_penalty) - learning_penalty

#     return {
#         'alive_mask': alive_mask,
#         'active_mask': active_mask,
#         'active_indices': list(active_indices),
#         'T_LC': T_LC,
#         'R_U': R_U,
#         'T_LU': T_LU,
#         'E_Cp': E_Cp,
#         'E_Cm': E_Cm,
#         'T_GA': float(T_GA),
#         'T_GU': float(T_GU),
#         'R_D': float(R_D),
#         'T_GD': float(T_GD),
#         'E_CpL': float(E_CpL),
#         'E_CL': float(E_CL),
#         'E_m_total': E_m_total,
#         'E_L_total': float(E_L_total),
#         'P_m_avg': P_m_avg,
#         'P_L_avg': float(P_L_avg),
#         'dead_count': dead_count,
#         'pen_dead': float(pen_dead),
#         'pen_no_active': float(pen_no_active),
#         'pen_e_m': float(pen_e_m),
#         'pen_e_l': float(pen_e_l),
#         'penalty': float(penalty),
#         'timeout_extra': float(timeout_extra),
#         'T_total': float(T_total),
#         'E_total': float(E_total),
#         'cost_time_norm': float(cost_time_norm),
#         'cost_energy_norm': float(cost_energy_norm),
#         'cost_penalty': float(cost_penalty),
#         'cost_normalized': float(cost_normalized),
#         'delta_acc': float(delta_acc),
#         'scaled_delta': float(scaled_delta),
#         'resource_cost': float(resource_cost),
#         'raw_ar_gain': float(raw_ar_gain),
#         'ar_gain': float(ar_gain),
#         'learning_penalty': float(learning_penalty),
#         'reward': float(reward),
#     }


# def _print_components(title, comp):
#     print(f"\n===== {title} =====")
#     print("[MASKS]")
#     print(f"- alive_mask: {comp['alive_mask'].astype(int).tolist()}")
#     print(f"- active_mask: {comp['active_mask'].astype(int).tolist()}")
#     print(f"- active_indices: {comp['active_indices']}")

#     print("[TIME COMPONENTS]")
#     print(f"- T_LC: {np.array2string(comp['T_LC'], precision=6)}")
#     print(f"- T_LU: {np.array2string(comp['T_LU'], precision=6)}")
#     print(f"- T_GA: {comp['T_GA']:.6f}")
#     print(f"- T_GU: {comp['T_GU']:.6f}")
#     print(f"- T_GD: {comp['T_GD']:.6f}")
#     print(f"- timeout_extra: {comp['timeout_extra']:.6f}")
#     print(f"- T_total: {comp['T_total']:.6f}")

#     print("[ENERGY / POWER COMPONENTS]")
#     print(f"- E_Cp: {np.array2string(comp['E_Cp'], precision=6)}")
#     print(f"- E_Cm: {np.array2string(comp['E_Cm'], precision=6)}")
#     print(f"- E_CpL: {comp['E_CpL']:.6f}")
#     print(f"- E_CL: {comp['E_CL']:.6f}")
#     print(f"- E_m_total: {np.array2string(comp['E_m_total'], precision=6)}")
#     print(f"- E_L_total: {comp['E_L_total']:.6f}")
#     print(f"- E_total: {comp['E_total']:.6f}")
#     print(f"- P_m_avg: {np.array2string(comp['P_m_avg'], precision=6)}")
#     print(f"- P_L_avg: {comp['P_L_avg']:.6f}")

#     print("[PENALTY BREAKDOWN]")
#     print(f"- dead_count: {comp['dead_count']}")
#     print(f"- pen_dead: {comp['pen_dead']:.6f}")
#     print(f"- pen_no_active: {comp['pen_no_active']:.6f}")
#     print(f"- pen_e_m: {comp['pen_e_m']:.6f}")
#     print(f"- pen_e_l: {comp['pen_e_l']:.6f}")
#     print(f"- total_penalty: {comp['penalty']:.6f}")

#     print("[COST + REWARD BREAKDOWN]")
#     print(f"- cost_time_norm: {comp['cost_time_norm']:.6f}")
#     print(f"- cost_energy_norm: {comp['cost_energy_norm']:.6f}")
#     print(f"- cost_penalty: {comp['cost_penalty']:.6f}")
#     print(f"- cost_normalized: {comp['cost_normalized']:.6f}")
#     print(f"- delta_acc: {comp['delta_acc']:.6f}")
#     print(f"- scaled_delta: {comp['scaled_delta']:.6f}")
#     print(f"- resource_cost: {comp['resource_cost']:.6f}")
#     print(f"- raw_ar_gain: {comp['raw_ar_gain']:.6f}")
#     print(f"- ar_gain: {comp['ar_gain']:.6f}")
#     print(f"- learning_penalty: {comp['learning_penalty']:.6f}")
#     print(f"- reward: {comp['reward']:.6f}")


# def manual_env_step_with_components(env_obj, action):
#     # 1) apply_action
#     delay_times, beta, force_mask, params = env_obj.apply_action(action)

#     # 2) FL round
#     acc, active_indices = env_obj._sync_run_fl_round(delay_times, beta, force_mask)

#     # 3) update learning state (giống step)
#     env_obj.current_acc = float(acc)
#     new_global_params = copy.deepcopy(get_model_params(env_obj.fl_sim.global_model))
#     env_obj.global_diff_sq = env_obj._compute_global_diff_sq(new_global_params)
#     env_obj.global_diff_norm = float(np.tanh(env_obj.global_diff_sq))
#     env_obj.prev_global_params = new_global_params

#     # 4) tính tay toàn bộ thành phần trước khi finalize_step xóa _phase_cache
#     comp = _calc_finalize_components(
#         env_obj,
#         active_indices=active_indices,
#         p_m=params['p_m'],
#         f_L=params['f_L'],
#         p_L=params['p_L'],
#     )

#     # 5) gọi finalize_step thật để đối chiếu output chuẩn env
#     obs, reward, done, truncated, info = env_obj.finalize_step(active_indices, params['p_m'], params['f_L'], params['p_L'])

#     return obs, reward, done, truncated, info, comp


# env_metrics = AUVSwarmEnv(M=9, debug=False, fl_sim=fl_sim_test)
# obs, _ = env_metrics.reset()

# actions = [
#     ("max", np.ones(env_metrics.action_space.shape, dtype=np.float32)),
#     ("min", -np.ones(env_metrics.action_space.shape, dtype=np.float32)),
#     ("random", env_metrics.action_space.sample().astype(np.float32)),
# ]

# for i, (name, act) in enumerate(actions, start=1):
#     obs, reward, terminated, truncated, info, comp = manual_env_step_with_components(env_metrics, act)
#     _print_components(f"STEP {i} | ACTION={name}", comp)

#     print("[FINALIZE OUTPUT INFO]")
#     for k in sorted(info.keys()):
#         v = info[k]
#         if isinstance(v, float):
#             print(f"- {k}: {v:.6f}")
#         else:
#             print(f"- {k}: {v}")

#     print("[ENV STATE AFTER FINALIZE]")
#     print(f"- current_step: {env_metrics.current_step}")
#     print(f"- current_acc: {env_metrics.current_acc:.6f}")
#     print(f"- prev_acc: {env_metrics.prev_acc:.6f}")
#     print(f"- global_diff_sq: {env_metrics.global_diff_sq:.6e}")
#     print(f"- global_diff_norm: {env_metrics.global_diff_norm:.6f}")
#     print(f"- battery_L: {env_metrics.battery_L:.6f}/{env_metrics.max_battery_L}")
#     print(f"- battery_m(min/mean/max): {env_metrics.battery_m.min():.6f} / {env_metrics.battery_m.mean():.6f} / {env_metrics.battery_m.max():.6f}")
#     print(f"- lazy_consecutive: {env_metrics.lazy_consecutive.tolist()}")

#     if terminated or truncated:

#         print("Episode kết thúc sớm, reset env để test tiếp...")print("\n✓ Hoàn tất in toàn bộ từng thành phần của finalize_step.")

#         obs, _ = env_metrics.reset()


===== STEP 1 | ACTION=max =====
[MASKS]
- alive_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1]
- active_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1]
- active_indices: [0, 1, 2, 3, 4, 5, 6, 7, 8]
[TIME COMPONENTS]
- T_LC: [0.1056 0.1056 0.1056 0.1056 0.1056 0.1056 0.1056 0.1056 0.1056]
- T_LU: [0.958398 0.958398 0.958398 0.958398 0.958398 0.958398 0.958398 0.958398
 0.958398]
- T_GA: 0.114768
- T_GU: 0.000000
- T_GD: 0.958398
- timeout_extra: 0.000000
- T_total: 2.137164
[ENERGY / POWER COMPONENTS]
- E_Cp: [0.08448 0.08448 0.08448 0.08448 0.08448 0.08448 0.08448 0.08448 0.08448]
- E_Cm: [0.19168 0.19168 0.19168 0.19168 0.19168 0.19168 0.19168 0.19168 0.19168]
- E_CpL: 0.102016
- E_CL: 0.191680
- E_m_total: [0.27616 0.27616 0.27616 0.27616 0.27616 0.27616 0.27616 0.27616 0.27616]
- E_L_total: 0.293696
- E_total: 2.779132
- P_m_avg: [0.129218 0.129218 0.129218 0.129218 0.129218 0.129218 0.129218 0.129218
 0.129218]
- P_L_avg: 0.137423
[PENALTY BREAKDOWN]
- dead_count: 0
- pen_dead: 0.000000
- pen_no_active: 0.0

In [25]:
# ─── Khởi tạo FLSimulator cho phần test env ───
print("Khởi tạo FLSimulator (thread mode) để test env...")
fl_sim_test = FLSimulator(num_workers=9, executor_type='thread', verbose=False)
print("✓ FLSimulator sẵn sàng.")


Khởi tạo FLSimulator (thread mode) để test env...
Loading MNIST for 9 workers...
✓ FLSimulator sẵn sàng.


### Cách 1: Dùng `check_env` của Stable-Baselines3

Hàm `check_env` cực kỳ nghiêm ngặt: kiểm tra Action/Observation Space có khớp dữ liệu thực tế, dtype đúng chuẩn, không tràn bounds.


In [26]:
from stable_baselines3.common.env_checker import check_env

# 1. Khởi tạo môi trường
env_test = AUVSwarmEnv(M=9, debug=False, fl_sim=fl_sim_test)

# 2. Chạy hàm kiểm tra
print("Đang kiểm tra môi trường...")
check_env(env_test, warn=True)
print("✅ Tuyệt vời! Môi trường không có lỗi cấu trúc (Syntax/Format).")


Đang kiểm tra môi trường...
✅ Tuyệt vời! Môi trường không có lỗi cấu trúc (Syntax/Format).


### Cách 2: Chạy thử với Random Agent (Sanity Check)

Cho môi trường chạy 2 episode với hành động hoàn toàn ngẫu nhiên để kiểm tra:
- Vòng lặp có kết thúc quá sớm không?
- Reward có ra giá trị kỳ dị (NaN, Infinity, âm vài triệu)?
- Pin có tụt âm?


In [27]:
env_rand = AUVSwarmEnv(M=9, debug=False, fl_sim=fl_sim_test)
episodes = 2

for ep in range(episodes):
    obs, info = env_rand.reset()
    done = False
    step_count = 0
    total_reward = 0
    
    print(f"\n--- Bắt đầu Episode {ep + 1} ---")
    while not done:
        # Lấy ngẫu nhiên 1 action hợp lệ trong Action Space
        random_action = env_rand.action_space.sample() 
        
        # Đưa action vào môi trường
        obs, reward, terminated, truncated, info = env_rand.step(random_action)
        done = terminated or truncated
        
        total_reward += reward
        step_count += 1
        
        # In thử log ở step đầu tiên để soi dữ liệu
        if step_count == 1:
            print(f"Action ngẫu nhiên: {random_action[:3]}...") # In 3 giá trị đầu
            print(f"Reward nhận được: {reward:.4f}")
            print(f"State trả về (shape): {obs.shape}")
            
    print(f"Kết thúc Episode {ep + 1} sau {step_count} steps. Tổng Reward: {total_reward:.2f}")



--- Bắt đầu Episode 1 ---
Action ngẫu nhiên: [-0.09414939 -0.872457    0.31238583]...
Reward nhận được: 99.9309
State trả về (shape): (33,)
Kết thúc Episode 1 sau 50 steps. Tổng Reward: 977.31

--- Bắt đầu Episode 2 ---
Action ngẫu nhiên: [-0.9087723  -0.83043087  0.69349986]...
Reward nhận được: 99.9671
State trả về (shape): (33,)
Kết thúc Episode 2 sau 50 steps. Tổng Reward: 1102.59


### Cách 3: Test thủ công các trường hợp cực đoan (Edge Cases)

Ép môi trường nhận action cực đoan để kiểm tra Penalty có kích hoạt đúng không:
- **Max Action**: Tất cả = +1 (Max Power, Max CPU)
- **Min Action**: Tất cả = -1 (Min Power, Min CPU)


In [28]:
import numpy as np

env_edge = AUVSwarmEnv(M=9, debug=True, fl_sim=fl_sim_test)
obs, info = env_edge.reset()

# Trường hợp 1: Ép hành động TỐI ĐA (Max Power, Max CPU)
max_action = np.ones(env_edge.action_space.shape, dtype=np.float32) 
obs, reward, term, trunc, info = env_edge.step(max_action)

print("\n====== Test Max Action ======")
print("- Reward:", reward)
print("- Penalty:", info.get('penalty', 0))
print("- Có bị kích hoạt Penalty không?:", "Có ✓" if info.get('penalty', 0) > 0 else "Không")
print("- Battery min follower:", info.get('battery_min', 'N/A'))

env_edge.reset()

# Trường hợp 2: Ép hành động TỐI THIỂU (Min Power, Min CPU)
min_action = -np.ones(env_edge.action_space.shape, dtype=np.float32)
obs, reward, term, trunc, info = env_edge.step(min_action)

print("\n====== Test Min Action ======")
print("- Reward:", reward)
print("- Penalty:", info.get('penalty', 0))
print("- Pin còn lại của Follower 0:", env_edge.battery_m[0])
print("- Battery min follower:", info.get('battery_min', 'N/A'))


[ENV-PENALTY] Follower energy threshold violated by 9 nodes. Penalty += 90.0

--- THONG KE STEP (Env Co-design) ---
Acc hien tai: 0.7726 | Delta acc: 0.772600 | Reward = 9.4037
Warmup scale: 0.125 | Scaled delta: 95.885
AR Gain (Reshaped): 99.40 | Raw Gain: 145.31
Penalty: 90.00 | Learning Penalty: 0.00
Tong thoi gian vong: T_total = 2.1499 s | E_total = 2.7791
-------------------------------------


====== Test Max Action ======
- Reward: 9.403665471950475
- Penalty: 90.0
- Có bị kích hoạt Penalty không?: Có ✓
- Battery min follower: 9.723840369411047

--- THONG KE STEP (Env Co-design) ---
Acc hien tai: 0.7756 | Delta acc: 0.775600 | Reward = 99.9999
Warmup scale: 0.125 | Scaled delta: 95.945
AR Gain (Reshaped): 100.00 | Raw Gain: 371.73
Penalty: 0.00 | Learning Penalty: 0.00
Tong thoi gian vong: T_total = 3.6833 s | E_total = 0.3764
-------------------------------------


====== Test Min Action ======
- Reward: 99.99993025662457
- Penalty: 0.0
- Pin còn lại của Follower 0: 9.96279458

### 💡 Lưu ý quan trọng: Normalize State & Reward

Khi train PPO, **bắt buộc** phải bọc env qua `VecNormalize` để:
- Chuẩn hóa Observation (battery `[0,10]`, accuracy `[0,1]`, ... → cùng scale)
- Chuẩn hóa Reward (tránh gradient bị bóp méo)

```python
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

env = DummyVecEnv([lambda: AUVSwarmEnv(M=9, fl_sim=fl_sim)])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)
```

> ⚠️ Action Space đã được chuẩn hóa về `[-1, 1]` trong `AUVSwarmEnv` (OK).  
> ⚠️ Observation Space đã khai báo `[0, 1]` nhưng cần `VecNormalize` để running mean/std chuẩn hóa động.


---
### ✅ Kết luận Test Env

Nếu cả 3 bước test trên đều pass:
- `check_env` ✅ không báo lỗi
- Random Agent chạy đủ episode ✅ không crash
- Edge cases ✅ penalty kích hoạt đúng

→ Bạn có thể tiến hành training PPO ở phần tiếp theo.


## 2B. Stable-Baselines3 Setup (Thay PPO tự code)

Phần này thay thế hoàn toàn mạng Actor-Critic, RolloutBuffer và PPO tự viết bằng thư viện **stable-baselines3**.

- Giữ nguyên môi trường `AUVSwarmEnv` đã định nghĩa ở trên.
- Dùng `stable_baselines3.PPO` với `MlpPolicy`.
- Thêm callback để log reward/cost theo step và theo episode.

In [29]:
# Cài gói nếu môi trường chưa có (chạy 1 lần nếu cần)
# %pip install stable-baselines3[extra]

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
import os

class TrainingLogCallback(BaseCallback):
    """Log reward/cost; gộp thông tin step và tổng hợp episode vào 1 file CSV update theo episode."""
    def __init__(self, print_every=50, detailed=False, verbose=0, log_file="training_log.csv", ep_log_file=None, starting_ep=0):
        super().__init__(verbose)
        self.print_every = int(print_every)
        self.detailed = bool(detailed)
        self.log_file = log_file
        
        # Xóa log cũ nếu có
        if os.path.exists(self.log_file):
            pass # Thay đổi: Không xóa log cũ nếu file đã tồn tại, mà ghi tiếp (append)
            # os.remove(self.log_file)
        else:
            with open(self.log_file, "w", encoding="utf-8") as f:
                f.write("Episode_ID,Step,Reward_Step,Cost_Raw,Delta_Accuracy,Penalty_Phys,AR_Gain,Ep_Length,Total_Reward,Total_Cost,Total_Time,Total_CommTime,Total_Energy,Final_Accuracy\n")
        
        # Biến tích lũy cho episode
        self.ep_reward = 0.0
        self.ep_cost = 0.0
        self.ep_time = 0.0
        self.ep_comm_time = 0.0
        self.ep_energy = 0.0
        self.ep_count = starting_ep
        
        # Buffer lưu data của từng step trong 1 episode
        self.step_buffer = []

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        rewards = self.locals.get("rewards", [0.0])
        dones = self.locals.get("dones", [False])
        
        reward_now = float(rewards[0])
        self.ep_reward += reward_now
        
        if infos:
            info = infos[0]
            # Các thông tin step
            cost_raw = float(info.get('cost', 0.0))
            delta_acc = float(info.get('delta_accuracy', 0.0))
            penalty = float(info.get('penalty', 0.0))
            ar_gain = reward_now + penalty # Xấp xỉ AR Gain thực tế
            
            # Lưu step data vào buffer tạm
            self.step_buffer.append({
                "step": self.num_timesteps,
                "reward": reward_now,
                "cost_raw": cost_raw,
                "delta_accuracy": delta_acc,
                "penalty": penalty,
                "ar_gain": ar_gain
            })
            
            # Cập nhật thông số lũy kế
            self.ep_cost += cost_raw
            self.ep_time += float(info.get('time', 0.0))
            self.ep_comm_time += float(info.get('comm_time', 0.0))
            self.ep_energy += float(info.get('energy', 0.0))
            
            # In ra console mỗi print_every step cho dễ theo dõi
            if self.n_calls % self.print_every == 0:
                print(f"[SB3] step={self.num_timesteps:5d} | reward={reward_now:7.4f} | cost={cost_raw:7.4f} | d_acc={delta_acc:8.5f} | pen={penalty:5.1f}")
            
            # KHI KẾT THÚC 1 EPISODE -> Flush buffer ghi tất cả các bước ra CSV 1 lượt
            if dones[0]:
                self.ep_count += 1
                final_acc = float(info.get('current_accuracy', 0.0))
                ep_length = len(self.step_buffer)
                
                with open(self.log_file, "a", encoding="utf-8") as f:
                    # Ghi từng step trong buffer
                    for s in self.step_buffer:
                        line = (f"{self.ep_count},{s['step']},{s['reward']:.4f},{s['cost_raw']:.4f},{s['delta_accuracy']:.5f},{s['penalty']:.1f},{s['ar_gain']:.4f},"
                                f"{ep_length},{self.ep_reward:.4f},{self.ep_cost:.4f},{self.ep_time:.4f},{self.ep_comm_time:.4f},{self.ep_energy:.4f},{final_acc:.4f}\n")
                        f.write(line)
                
                # Reset lại biến cho episode tiếp theo
                self.ep_reward = 0.0
                self.ep_cost = 0.0
                self.ep_time = 0.0
                self.ep_comm_time = 0.0
                self.ep_energy = 0.0
                self.step_buffer = []

        return True

print("✓ Stable-Baselines3 TrainingLogCallback gộp 1 file update theo mỗi ep.")

✓ Stable-Baselines3 TrainingLogCallback gộp 1 file update theo mỗi ep.


## 2C. Huấn luyện PPO bằng Stable-Baselines3

Huấn luyện trực tiếp trên `AUVSwarmEnv` thông qua API chuẩn Gymnasium (`reset/step`).

Các log chính:
- Callback theo số bước (`step`, `reward`, `cost_raw`)
- Kết quả mean reward sau khi train
- Lưu model `.zip` để dùng lại khi đánh giá

In [ ]:
import os
import torch
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList

def train_ppo_sb3(
    M=9,
    model_dir="model",
    log_dir="logs",
    total_timesteps=10000,
    seed=42,
    reward_debug=False,
    callback_print_every=50,
    save_freq=1000, # lưu sau mỗi 1000 steps (có thể tùy chỉnh)
    fl_sim=None,
    use_tensorboard=True,
    resume_checkpoint=None,
    starting_step=0,
    starting_ep=0
):
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs("checkpoints", exist_ok=True)

    if fl_sim is None:
        raise ValueError("fl_sim is required. Example: FLSimulator(num_workers=M, executor_type='thread')")

    # reward_debug=True: giu log callback chi tiet; env verbose hien tai ke thua 100% logic co-design.
    env_cls = AUVSwarmEnvVerbose if bool(reward_debug) else AUVSwarmEnv
    env = env_cls(M=M, debug=bool(reward_debug), fl_sim=fl_sim)
    env = Monitor(env)

    tb_log_dir = log_dir if bool(use_tensorboard) else None

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        load_path = resume_checkpoint
        # Kaggle tự động giải nén file .zip thành folder. Ta cần nén lại để PPO.load() đọc được.
        if os.path.isdir(resume_checkpoint):
            import shutil
            print(f"Directory detected (Kaggle auto-unzip). Zipping {resume_checkpoint} to temp file...")
            temp_zip = "/kaggle/working/temp_resume_model"
            shutil.make_archive(temp_zip, 'zip', resume_checkpoint)
            load_path = temp_zip + ".zip"
            
        print(f"Resuming model from {load_path}...")
        model = PPO.load(load_path, env=env, tensorboard_log=tb_log_dir)
        reset_num_timesteps = False
        # Thiết lập lại số bước nếu thư viện không tự lấy đúng
        if starting_step > 0:
            model.num_timesteps = starting_step
    else:
        model = PPO(
            policy="MlpPolicy",  # dùng MLP cho state vector liên tục của môi trường
            env=env,  # môi trường Gym/Gymnasium cung cấp sample action/reward
            learning_rate=3e-4,  # tốc độ học của optimizer
            n_steps=256,  # số bước rollout trước mỗi lần update policy
            batch_size=64,  # số mẫu trong mỗi mini-batch khi tối ưu
            n_epochs=10,  # số epoch lặp lại trên cùng rollout buffer
            gamma=0.99,  # hệ số chiết khấu reward tương lai
            gae_lambda=0.95,  # tham số GAE cân bằng bias/variance
            clip_range=0.2,  # biên clip PPO để giới hạn thay đổi policy
            ent_coef=0.01,  # hệ số entropy bonus khuyến khích khám phá
            vf_coef=0.5,  # trọng số loss của value function
            max_grad_norm=0.5,  # gradient clipping để tránh nổ gradient
            tensorboard_log=tb_log_dir,  # thư mục ghi log TensorBoard
            verbose=1,  # mức độ in log khi train
            seed=seed,  # seed cố định để tái lập kết quả
            policy_kwargs=dict(net_arch=[256, 128]),  # kiến trúc 2 hidden layers cho policy/value network
        )
        reset_num_timesteps = True

    log_callback = TrainingLogCallback(
        print_every=callback_print_every,
        detailed=bool(reward_debug),
        log_file="training_log.csv",
        starting_ep=starting_ep
    )
    
    checkpoint_callback = CheckpointCallback(
        save_freq=save_freq,
        save_path='checkpoints/',
        name_prefix=f'ppo_auv_M{M}'
    )
    
    callback_list = CallbackList([log_callback, checkpoint_callback])
    
    model.learn(total_timesteps=total_timesteps, callback=callback_list, progress_bar=False, reset_num_timesteps=reset_num_timesteps)

    model_path = os.path.join(model_dir, f"sb3_ppo_auv_M{M}")
    model.save(model_path)

    mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=5, deterministic=True)
    print(f"✓ Saved model: {model_path}.zip")
    print(f"✓ Eval mean reward: {mean_reward:.4f} ± {std_reward:.4f}")

    return model, model_path, (mean_reward, std_reward)

print("✓ train_ppo_sb3 defined.")

✓ train_ppo_sb3 defined.


## 2D. Huấn luyện PPO (Fresh Start / Resume)

Cell dưới đây khởi tạo `FLSimulator` và chạy training PPO.
- Để train mới từ đầu: bỏ trống `resume_checkpoint`.
- Để resume: truyền đường dẫn checkpoint vào `resume_checkpoint`.


In [31]:
# ─── Khởi tạo FLSimulator ───
print("Khởi tạo FLSimulator (thread mode) cho training FL+RL...")
fl_simulator = FLSimulator(num_workers=9, executor_type='thread', verbose=False)

# ─── Config ───
TOTAL_TIMESTEPS = 10000   # Tổng số timesteps cần train
MODEL_DIR = "model"       # Thư mục lưu model
LOG_DIR = "logs"          # Thư mục lưu log / tensorboard
RESUME_CHECKPOINT = None  # Đặt đường dẫn .zip nếu muốn resume, None = train mới
STARTING_STEP = 0        # Timestep bắt đầu (chỉ dùng khi resume)
STARTING_EP = 0          # Episode bắt đầu (chỉ dùng khi resume, cho CSV log)

print(f"Bắt đầu train PPO | timesteps={TOTAL_TIMESTEPS} | resume={RESUME_CHECKPOINT is not None}")

sb3_model, sb3_model_path, sb3_eval = train_ppo_sb3(
    M=9,
    total_timesteps=TOTAL_TIMESTEPS,
    model_dir=MODEL_DIR,
    log_dir=LOG_DIR,
    seed=42,
    reward_debug=False,
    callback_print_every=50,
    save_freq=500,
    fl_sim=fl_simulator,
    resume_checkpoint=RESUME_CHECKPOINT,
    starting_step=STARTING_STEP,
    starting_ep=STARTING_EP,
)

print(f"\n✅ Hoàn tất train. Model: {sb3_model_path}.zip")
print(f"✅ Mean reward: {sb3_eval[0]:.4f} ± {sb3_eval[1]:.4f}")


Khởi tạo FLSimulator (thread mode) cho training FL+RL...
Loading MNIST for 9 workers...
Bắt đầu train PPO | timesteps=10000 | resume=False
Using cpu device
Wrapping the env in a DummyVecEnv.
Logging to logs\PPO_2
[SB3] step=   50 | reward= 4.8429 | cost= 4.5160 | d_acc= 0.00020 | pen=  0.0
[SB3] step=  100 | reward=13.2498 | cost= 2.6205 | d_acc= 0.00030 | pen=  0.0
[SB3] step=  150 | reward= 3.1552 | cost= 3.4708 | d_acc=-0.00050 | pen=  0.0


KeyboardInterrupt: 

In [ ]:
# Cứu Model và Rollout 1 episode để vẽ Reward/Cost (thay cho hàm Loss của RL)

# Đảm bảo lưu đúng Model hiện tại một lần nữa (trường hợp user muốn thủ công)
save_path = f"model/ppo_auv_swarm_M{fl_simulator.num_workers}_final.pth"
sb3_model.save(save_path)
print(f"Đã lưu thủ công mô hình tại: {save_path}")

def rollout_once(model, fl_sim, M=9, max_steps=50):
    # Fix: Cần truyền fl_sim vào như trong lúc train để tránh lỗi thiếu FLSimulator
    env = AUVSwarmEnv(M=M, debug=False, fl_sim=fl_sim)
    obs, _ = env.reset()

    rewards = []
    costs = []

    for _ in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        rewards.append(float(reward))
        costs.append(float(info.get("cost_normalized", info.get("cost", np.nan))))
        if done or truncated:
            break

    return np.array(rewards), np.array(costs)

# Sử dụng fl_simulator đang hoạt động
r_hist, c_hist = rollout_once(sb3_model, fl_sim=fl_simulator, M=9, max_steps=50)

# Vẽ biểu đồ Reward (đóng vai trò Reward trend / Gain) và Cost (đóng vai trò Loss vật lý)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(r_hist, color="tab:blue", marker='o', markersize=4)
axes[0].set_title("Reward (AR Gain) per step (1 episode)")
axes[0].set_xlabel("Step (FL Round)")
axes[0].set_ylabel("Reward")
axes[0].grid(alpha=0.3)

axes[1].plot(c_hist, color="tab:red", marker='o', markersize=4)
axes[1].set_title("Normalized Cost per step (1 episode)")
axes[1].set_xlabel("Step (FL Round)")
axes[1].set_ylabel("Cost")
axes[1].grid(alpha=0.3)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/rollout_eval_reward_cost.png", dpi=300)
plt.show()
print("Đã lưu biểu đồ đánh giá tại figures/rollout_eval_reward_cost.png")

## 2E. Evaluation SB3 PPO và Baselines

So sánh `sb3_ppo` với `max_power`, `min_power`, `random` theo chi phí trung bình trên nhiều episode.

In [ ]:
def evaluate_scheme(M=9, scheme="sb3_ppo", model=None, n_episodes=5, max_steps=200, fl_sim=None):
    ep_costs = []
    ep_rewards = []

    for _ in range(n_episodes):
        env = AUVSwarmEnv(M=M, debug=False, fl_sim=fl_sim)
        obs, _ = env.reset()

        total_cost = 0.0
        total_reward = 0.0

        for _ in range(max_steps):
            if scheme == "sb3_ppo":
                action, _ = model.predict(obs, deterministic=True)
            elif scheme == "max_power":
                action = np.ones(env.action_space.shape[0], dtype=np.float32)
            elif scheme == "min_power":
                action = -np.ones(env.action_space.shape[0], dtype=np.float32)
            else:
                action = env.action_space.sample()

            obs, reward, done, truncated, info = env.step(action)
            total_reward += float(reward)
            total_cost += float(info.get("cost", 0.0))

            if done or truncated:
                break

        ep_costs.append(total_cost)
        ep_rewards.append(total_reward)

    return float(np.mean(ep_costs)), float(np.mean(ep_rewards))

schemes = ["sb3_ppo", "max_power", "min_power", "random"]
results = {}    

for sc in schemes:
    mean_cost, mean_reward = evaluate_scheme(
        M=9,
        scheme=sc,
        model=sb3_model,
        n_episodes=5,
        max_steps=200,
        fl_sim=fl_simulator,
    )
    results[sc] = {"cost": mean_cost, "reward": mean_reward}
    print(f"{sc:10s} | mean_cost={mean_cost:.4f} | mean_reward={mean_reward:.4f}")

labels = list(results.keys())
cost_vals = [results[k]["cost"] for k in labels]
reward_vals = [results[k]["reward"] for k in labels]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, cost_vals, color=["tab:blue", "tab:orange", "tab:green", "tab:red"])
axes[0].set_title("Mean Cost by Scheme")
axes[0].set_ylabel("Cost")
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(labels, reward_vals, color=["tab:blue", "tab:orange", "tab:green", "tab:red"])
axes[1].set_title("Mean Reward by Scheme")
axes[1].set_ylabel("Reward")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

---
## Tổng kết (phiên bản SB3)

| Thành phần | Trạng thái |
|:-----------|:-----------|
| FL Core (tự code) | Giữ nguyên như notebook gốc |
| RL Environment `AUVSwarmEnv` | Giữ nguyên code sẵn có |
| PPO Agent | Thay bằng `stable_baselines3.PPO` |
| Training loop | Dùng `model.learn(...)` + callback log |
| Evaluation | So sánh SB3 PPO với các baseline `max_power/min_power/random` |

Notebook này là bản sao của notebook gốc, chỉ thay phần PPO/train/log phía dưới môi trường bằng Stable-Baselines3.